# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2 Dataset: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` and required Python libraries are installed
!pip install mlcroissant pandas matplotlib

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Here we enumerate the record sets defined in the metadata and explore their fields and columns. All references to entities are via their `@id`.

In [ ]:
# List all record sets and their fields using @id

# Get all record sets from the metadata
rs_list = []
if hasattr(metadata, 'record_sets'):
    rs_list = metadata.record_sets
elif hasattr(metadata, 'record_set'):
    rs_list = metadata.record_set

# If no record sets found, stop here
if not rs_list:
    print("No record sets found in the schema.")
else:
    for rs in rs_list:
        print(f"RecordSet @id: {rs['@id']}")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print('  Fields:')
        for field in fields:
            print(f"    Field @id: {field['@id']} (name: {field.get('name','')})")
        columns = rs.get('column', [])
        if columns:
            if isinstance(columns, dict):
                columns = [columns]
            print('  Columns:')
            for col in columns:
                print(f"    Column @id: {col['@id']} (name: {col.get('name','')})")
    print('-' * 40)

Let's print a sample of records from the main record set.

_You should update `main_record_set_id` below with the `@id` of the record set you want to analyze (see previous cell's output)._

In [ ]:
# You will need to insert the correct record set '@id' here.
main_record_set_id = 'cr:RecordSet_1'  # <-- Replace with actual @id from the output above

try:
    for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
        print(f"Sample record {i}: {record}")
        if i >= 2:
            break
except Exception as e:
    print(f"Could not load records for record set {main_record_set_id}: {e}")

## 3. Data Extraction
Load data from selected record set(s) into a DataFrame for analysis.

**Note:** Edit the list below to include all desired record set `@id`s you found in the Data Overview.

In [ ]:
# List your record set @id(s) below from the overview above
record_sets = ['cr:RecordSet_1']  # <-- Fill with all discovered record_set @ids

dataframes = {}
for record_set in record_sets:
    try:
        records = list(dataset.records(record_set=record_set))
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"\nRecord Set: {record_set}")
        print("Columns:", df.columns.tolist())
        print(df.head())
    except Exception as e:
        print(f"Could not load data for record set {record_set}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing: filter, normalize, group, etc.

**Instructions:**
  - Choose a numeric field (its `@id`) from the columns listed above.
  - Choose a grouping field (`@id`), e.g., a categorical or string field.

Be sure to use the exact DataFrame key and column name corresponding to their `@id`.

In [ ]:
# Set up correct record set and numeric/group field IDs here.
record_set_id = 'cr:RecordSet_1'  # <-- should match the main record set above
numeric_field = 'cr:Field_coverage_percentage'  # <-- Replace with a numeric field @id from Data Overview
group_field = 'cr:Field_sample_id'  # <-- Replace with a string/categorical field @id

# Filtering: only rows where the numeric field > threshold
threshold = 10
df = dataframes[record_set_id]
if numeric_field in df.columns:
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a specified field and take the mean
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
    else:
        print(f"Grouping field '{group_field}' not found in DataFrame columns.")
else:
    print(f"Numeric field '{numeric_field}' not found in DataFrame columns.")

## 5. Visualization
Visualize selected numeric fields and their relationships.

Below we draw a histogram and a group-wise box plot. Edit the fields as needed.

In [ ]:
# Plot a histogram of the numeric field
if numeric_field in df.columns:
    plt.figure(figsize=(8, 5))
    df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

# Boxplot by group field
if group_field in df.columns:
    df.boxplot(column=numeric_field, by=group_field, figsize=(10, 6))
    plt.title(f"{numeric_field} by {group_field}")
    plt.suptitle("")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 Croissant dataset [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

- We outlined all available record sets and their fields by their unique `@id`.
- Data was extracted and loaded into DataFrames for further exploration.
- We performed basic filtering, normalization, and grouped analysis on numeric fields (referencing only by their `@id`).
- Visualizations provided insight into field distributions and relationships.

For further research, update the fields and record set IDs as the dataset schema evolves, and apply additional domain-specific analysis as needed.